In [1]:
!pip3 install bitsandbytes peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 37.6 MB/s eta 0:00:00


## **Retrieving & Processing Dataset**

In [2]:
!git clone https://github.com/Lavender105/RSGPT.git

Cloning into 'RSGPT'...
remote: Enumerating objects: 3259, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 3259 (delta 10), reused 0 (delta 0), pack-reused 3241 (from 2)
Receiving objects: 100% (3259/3259), 1.13 GiB | 45.64 MiB/s, done.
Resolving deltas: 100% (40/40), done.
Updating files: 100% (3181/3181), done.


In [3]:
import json
with open("/kaggle/working/RSGPT/dataset/RSICap/captions.json",'r') as f:
  single=json.load(f)

with open("/kaggle/working/RSGPT/dataset/RSIEval/annotations.json",'r') as f:
  multiple=json.load(f)

In [4]:
updated_single=single['annotations']

In [9]:
filenames=[multiple['annotations'][i]['filename'] for i in range(len(multiple['annotations']))]
qa_pairs=[multiple['annotations'][i]['qa_pairs'] for i in range(len(multiple['annotations']))]

In [14]:
updated_multiple = []
for i in range(100):
  for j in range(len(qa_pairs[i])):
    temp_qa = {}
    temp_qa['filename'] = filenames[i]
    temp_qa['question'] = qa_pairs[i][j]['question']
    temp_qa['answer'] = qa_pairs[i][j]['answer']
    updated_multiple.append(temp_qa)

In [8]:
len(updated_multiple)

943

In [11]:
len(updated_single)

2585

In [9]:
updated_single[1499]

{'image_id': '1499',
 'filename': 'P2672_0015.png',
 'text_input': 'Describe this image in detail.',
 'text_output': 'This is an aerial image that shows both water and land. A grass field is located on the left side of the image. A parking lot with twelve cars is located in the middle of the grass field. A water area is located on the right side of the grass field, with two harbors. A boat is parked next to the lower harbor. Another grass field is located on the right side of the water area. There is also another water area on the right side of the grass field.'}

In [10]:
updated_multiple[0]

{'filename': 'P1384_0054.png',
 'question': 'Does this image contains trees?',
 'answer': 'Yes.'}

In [12]:
import os
single_img_path="/kaggle/working/RSGPT/dataset/RSICap/images"
single_images=[]
for item in updated_single:
    img_path = os.path.join(single_img_path, item['filename'])
    single_images.append({
        "image":img_path,
        "question":item['text_input'],
        "answer": item['text_output']
    })

In [13]:
single_images[0]

{'image': '/kaggle/working/RSGPT/dataset/RSICap/images/P0378_0001.png',
 'question': 'Take a look at this image and describe what you notice.',
 'answer': "This is an aerial image showing a parking lot with a high resolution. In the parking lot, there are many vehicles, including cars, trucks, and buses. Specifically, thirty-seven large-vehicles and fourteen small-vehicles can be observed in the image. Some of them are parked in rows, and others are parked in random positions. In the image's bottom-right, the trucks and buses are parked in a line, with some facing the same direction and some facing the opposite direction. Some of the trucks and buses are white, while others are yellow. Overall, the parking lot is filled with a variety of vehicles. There is a structure in the image's top left and two roadside green belts at the bottom of the image. The image shows a sunny day since we can see the shadow of the vehicles and trees."}

In [14]:
len(single_images)

2585

In [15]:
multiple_img_path="/kaggle/working/RSGPT/dataset/RSIEval/images"
multiple_images=[]
for item in updated_multiple:
    img_path = os.path.join(multiple_img_path, item['filename'])
    multiple_images.append({
        "image":img_path,
        "question":item['question'],
        "answer": item['answer']
    })

In [16]:
multiple_images[0]

{'image': '/kaggle/working/RSGPT/dataset/RSIEval/images/P1384_0054.png',
 'question': 'Does this image contains trees?',
 'answer': 'Yes.'}

In [17]:
len(multiple_images)

943

In [18]:
data=single_images+multiple_images

In [19]:
len(data)

3528

## **Creating HuggingFace dataset**

In [20]:
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
import os
from PIL import Image

In [21]:
dataset = Dataset.from_list(data)

In [22]:
dataset = dataset.shuffle(seed=42) 

In [31]:
dataset[444]

{'image': '/kaggle/working/RSGPT/dataset/RSIEval/images/P1384_0147.png',
 'question': 'Where is the water in the image?',
 'answer': 'Top left corner.'}

In [32]:
from datasets import DatasetDict

# First split: train + temp (test+val)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

# Second split: test + validation
temp = dataset["test"].train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    "train": dataset["train"],
    "validation": temp["train"],
    "test": temp["test"],
})


In [37]:
from datasets import Image

features = Features({
    "image": Image(),
    "question": Value("string"),
    "answer": Value("string"),
    "type": Value("string"),
})


In [38]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 2822
    })
    validation: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 353
    })
    test: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 353
    })
})

In [39]:
dataset = dataset.cast_column("image", Image())

In [40]:
e1=dataset['train'][0]
e1

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=512x512>,
 'question': 'Is there a ground-track-field in the image?',
 'answer': 'Yes.'}

In [41]:
e1['answer']

'Yes.'

In [42]:
dataset['train']

Dataset({
    features: ['image', 'question', 'answer'],
    num_rows: 2822
})

## **Model**

In [43]:
import os
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
import torch
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

import warnings
warnings.filterwarnings("ignore")

2025-12-31 14:07:12.872512: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767190033.062024     105 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767190033.115490     105 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767190033.551721     105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767190033.551768     105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767190033.551771     105 computation_placer.cc:177] computation placer alr

In [44]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [45]:
system_message = """You are a highly advanced Vision Language Model (VLM), specialized in analyzing, describing, and interpreting visual data. 
Your task is to process and extract meaningful insights from images, videos, and visual patterns, 
leveraging multimodal understanding to provide accurate and contextually relevant information."""

def format_data(sample):
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_message}],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": sample["image"],
                },
                {
                    "type": "text",
                    "text": sample["question"],
                },
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["answer"]}],
        },
    ]

In [46]:
train_dataset = [format_data(sample) for sample in dataset['train']]
eval_dataset = [format_data(sample) for sample in dataset['validation']]
test_dataset = [format_data(sample) for sample in dataset['test']]

In [47]:
train_dataset[0]

[{'role': 'system',
  'content': [{'type': 'text',
    'text': 'You are a highly advanced Vision Language Model (VLM), specialized in analyzing, describing, and interpreting visual data. \nYour task is to process and extract meaningful insights from images, videos, and visual patterns, \nleveraging multimodal understanding to provide accurate and contextually relevant information.'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=512x512>},
   {'type': 'text', 'text': 'Is there a ground-track-field in the image?'}]},
 {'role': 'assistant', 'content': [{'type': 'text', 'text': 'Yes.'}]}]

In [48]:
MODEL_ID="Qwen/Qwen2-VL-2B-Instruct"

In [49]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, 
    device_map="auto", 
    quantization_config=bnb_config,
    use_cache=False
    )
processor = Qwen2VLProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [50]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

print(f"Before adapter parameters: {model.num_parameters()}")
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters() # After LoRA trainable parameters increases. Since we add adapter.

Before adapter parameters: 2208985600
trainable params: 1,089,536 || all params: 2,210,075,136 || trainable%: 0.0493


In [51]:
training_args = SFTConfig(
    output_dir="./output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    logging_steps=100,
    eval_steps=100,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=100,
    max_length=128,
    metric_for_best_model="eval_loss",
    load_best_model_at_end=True,
    max_grad_norm=1.0,
    warmup_steps=0,
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns = False,
    optim="paged_adamw_32bit",
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [52]:
collate_sample = [train_dataset[0], train_dataset[1]] # for batch size 2.

def collate_fn(examples):
    texts = [processor.apply_chat_template(example, tokenize=False) for example in examples]
    image_inputs = [example[1]["content"][0]["image"] for example in examples]

    batch = processor(
        text=texts, images=image_inputs, return_tensors="pt", padding=True
    )
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = batch["input_ids"]

    return batch

collated_data = collate_fn(collate_sample)
print(list(collated_data.keys()))  # dict_keys(['input_ids', 'attention_mask', 'pixel_values', 'labels'])

['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw', 'labels']


In [53]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
    processing_class=processor.tokenizer,
)

In [57]:
print("Training")
trainer.train(resume_from_checkpoint="/kaggle/working/output/checkpoint-400")
print("-"*30)

Training


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,5.499900,5.489242,5.900090,551961.000000,0.277247
600,5.471200,5.485664,5.892722,644165.000000,0.276735
700,5.442200,5.481030,5.889444,736535.000000,0.276770
800,5.529100,5.478634,5.893967,827566.000000,0.277605
900,5.436800,5.475750,5.882730,920435.000000,0.277877
1000,5.484700,5.474640,5.886973,1012170.000000,0.277674
1100,5.515000,5.472956,5.883661,1103737.000000,0.278073
1200,5.454000,5.471591,5.880027,1195517.000000,0.278416
1300,5.471200,5.469101,5.871600,1287501.000000,0.278602
1400,5.430200,5.467699,5.873107,1379357.000000,0.278793


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

------------------------------


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Training


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,7.753500,5.613815,6.063721,90954.000000,0.269373
200,5.639400,5.545677,5.977644,182233.000000,0.273158
300,5.551300,5.509043,5.941637,274742.000000,0.275617
400,5.531200,5.496802,5.917671,366633.000000,0.276188


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [68]:
print("-"*30)
print("Evaluation")
metric = trainer.evaluate()
print(metric)
print("-"*30)

------------------------------
Evaluation


{'eval_loss': 5.460772514343262, 'eval_runtime': 671.4757, 'eval_samples_per_second': 0.526, 'eval_steps_per_second': 0.264, 'eval_entropy': 5.866466767370364, 'eval_num_tokens': 2685177.0, 'eval_mean_token_accuracy': 0.279727559726117, 'epoch': 2.0}
------------------------------


In [58]:
trainer.save_model(f"{training_args.output_dir}/new")

In [71]:
import gc
import time

# https://huggingface.co/learn/cookbook/en/fine_tuning_vlm_trl
def clear_memory():
    if "inputs" in globals():
        del globals()["inputs"]
    if "model" in globals():
        del globals()["model"]
    if "processor" in globals():
        del globals()["processor"]
    if "trainer" in globals():
        del globals()["trainer"]
    if "peft_model" in globals():
        del globals()["peft_model"]
    if "bnb_config" in globals():
        del globals()["bnb_config"]
    time.sleep(2)

    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


clear_memory()

GPU allocated memory: 0.01 GB
GPU reserved memory: 0.39 GB


In [60]:
with open("/kaggle/working/output/new/README.md", "w") as f:
    f.write("""---
base_model: Qwen/Qwen2-VL-2B-Instruct
library_name: peft
---

## Qwen2-VL-2B LoRA Adapter

This repository contains a LoRA adapter fine-tuned from **Qwen/Qwen2-VL-2B-Instruct**.
""")


In [62]:
from huggingface_hub import login
login()

In [66]:
from huggingface_hub import HfApi
api = HfApi()

api.create_repo(
    "qwen2vl-Lora-new",
    repo_type="model",
    exist_ok=True
)

RepoUrl('https://huggingface.co/Abdullah-123/qwen2vl-Lora-new', endpoint='https://huggingface.co', repo_type='model', repo_id='Abdullah-123/qwen2vl-Lora-new')

In [70]:
api.upload_folder(
    folder_path="/kaggle/working/output/new",
    repo_id="Abdullah-123/qwen2vl-Lora-new",
    repo_type="model"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Abdullah-123/qwen2vl-Lora-new/commit/5d64dcc0ec1fe29d0dd7ce4058051741fd7d653b', commit_message='Upload folder using huggingface_hub', commit_description='', oid='5d64dcc0ec1fe29d0dd7ce4058051741fd7d653b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Abdullah-123/qwen2vl-Lora-new', endpoint='https://huggingface.co', repo_type='model', repo_id='Abdullah-123/qwen2vl-Lora-new'), pr_revision=None, pr_num=None)

In [72]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    use_cache=True
    )
processor = Qwen2VLProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [73]:
print(f"Before adapter parameters: {model.num_parameters()}")
model.load_adapter("Abdullah-123/qwen2vl-Lora-new")
print(f"After adapter parameters: {model.num_parameters()}")

Before adapter parameters: 2208985600


adapter_config.json:   0%|          | 0.00/858 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.37M [00:00<?, ?B/s]

After adapter parameters: 2210075136
